In [328]:
import numpy as np
import pandas as pd

In [329]:
df=pd.read_csv('Bengaluru_House_Data.csv')
df

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00
...,...,...,...,...,...,...,...,...,...
13315,Built-up Area,Ready To Move,Whitefield,5 Bedroom,ArsiaEx,3453,4.0,0.0,231.00
13316,Super built-up Area,Ready To Move,Richards Town,4 BHK,NaN,3600,5.0,NaN,400.00
13317,Built-up Area,Ready To Move,Raja Rajeshwari Nagar,2 BHK,Mahla T,1141,2.0,1.0,60.00
13318,Super built-up Area,18-Jun,Padmanabhanagar,4 BHK,SollyCl,4689,4.0,1.0,488.00


In [330]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  object 
 1   availability  13320 non-null  object 
 2   location      13319 non-null  object 
 3   size          13304 non-null  object 
 4   society       7818 non-null   object 
 5   total_sqft    13320 non-null  object 
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), object(6)
memory usage: 936.7+ KB


In [331]:
for column in df.columns:
    print(df[column].value_counts())
    print("*"*20)

area_type
Super built-up  Area    8790
Built-up  Area          2418
Plot  Area              2025
Carpet  Area              87
Name: count, dtype: int64
********************
availability
Ready To Move    10581
18-Dec             307
18-May             295
18-Apr             271
18-Aug             200
                 ...  
15-Dec               1
16-Oct               1
16-Jan               1
14-Nov               1
14-Jul               1
Name: count, Length: 81, dtype: int64
********************
location
Whitefield                 540
Sarjapur  Road             399
Electronic City            302
Kanakpura Road             273
Thanisandra                234
                          ... 
Park View Layout             1
Xavier Layout                1
Air View Colony              1
akshaya nagar t c palya      1
mvj engineering college      1
Name: count, Length: 1305, dtype: int64
********************
size
2 BHK         5199
3 BHK         4310
4 Bedroom      826
4 BHK          591
3 Bedroom 

In [332]:
df.isna().sum()

area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64

In [333]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df)
profile.to_file('DataAnalysis.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 9/9 [00:00<00:00, 15.16it/s]
/home/kernel00/miniconda3/envs/AI/lib/python3.11/site-packages/seaborn/matrix.py:260: FutureWarning: Format strings passed to MaskedConstant are ignored, but in future may error or produce different behavior
  annotation = ("{:" + self.fmt + "}").format(val)
/home/kernel00/miniconda3/envs/AI/lib/python3.11/site-packages/ydata_profiling/model/missing.py:132: UserWarning: There was an attempt to generate the Heatmap missing values diagrams, but this failed.
        To hide this warning, disable the calculation
        (using `df.profile_report(missing_diagrams={"Heatmap": False}`)
        If this is problematic for your use case, please report this as an issue:
        https://github.com/ydataai/ydata-profiling/issues
        (include the error message: 'could not convert string to float: '--'')
  warnings.warn(


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

## Droppping irrelevent columns

In [334]:
# society has 5502 (41.3%) missing values and also has 2688 distinct value so droppping it

df=df.drop(columns=['society'])

In [335]:
# area typep has 0.518 corealtion with size so area type is redundent also area type is weakly corelated with price (0.107) so dropping area type

# also balcony is weakly corealted with price (0.042) and balcony has 0.299 corealation with size. Thus size can explain area type and balcony to some extent so removing both of the redundent values as they both contribute very less to the price.
df=df.drop(columns=['area_type','balcony'])

In [336]:
# df['size'] has inconsistent data like 2 BHK and 2 bedroom. so extracting just digit out of it and finding out it's corelation with the price column.

df['bhk'] = df['size'].str.extract(r'(\d+)').astype(float)
df['bhk'].corr(df['price'])
df.drop(columns=['size'],inplace=True)

# The r prefix makes it a raw string
# ( )	A capture group. It marks the part of the match you want back. extract returns only what’s inside the parentheses, and raises an error if there are none.
# \d means a digit and + means one or more

In [337]:
df['availability'].value_counts()

availability
Ready To Move    10581
18-Dec             307
18-May             295
18-Apr             271
18-Aug             200
                 ...  
15-Dec               1
16-Oct               1
16-Jan               1
14-Nov               1
14-Jul               1
Name: count, Length: 81, dtype: int64

In [338]:
# This column has class imbalance (79%).A column that's 79% one value carries very little information (low entropy).
# a listing available "18-Dec" vs "18-Jan" isn't obviously going to be priced differently — this isn't like premium societies commanding higher prices. however the flat that are ready to move may have higher price.

In [339]:
# before dropping this column,let's built 'ready to move' column and let's see its corelation with the price 
df['ready_to_move'] = (df['availability'] == 'Ready To Move').astype(int)
df['ready_to_move'].corr(df['price'])


0.03448293454298826

In [340]:
df.drop(columns=['ready_to_move'],inplace=True)
#  0.034 is essentially no relationship, even weaker than balcony's 0.042. Dropping it is well justified — and unlike society, this isn't a "practical tradeoff despite real signal" situation. There's just no meaningful relationship here to lose.

df.drop(columns=['availability'],inplace=True)

In [341]:
df.describe()

,bath,price,bhk
count,13247.000000,13320.000000,13304.000000
mean,2.692610,112.565627,2.803743
std,1.341458,148.971674,1.294974
min,1.000000,8.000000,1.000000
25%,2.000000,50.000000,2.000000
50%,2.000000,72.000000,3.000000
75%,3.000000,120.000000,3.000000
max,40.000000,3600.000000,43.000000


## Filling missing values

In [342]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   location    13319 non-null  object 
 1   total_sqft  13320 non-null  object 
 2   bath        13247 non-null  float64
 3   price       13320 non-null  float64
 4   bhk         13304 non-null  float64
dtypes: float64(3), object(2)
memory usage: 520.4+ KB


In [343]:
df=df.dropna(subset=['location'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13319 entries, 0 to 13319
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   location    13319 non-null  object 
 1   total_sqft  13319 non-null  object 
 2   bath        13246 non-null  float64
 3   price       13319 non-null  float64
 4   bhk         13303 non-null  float64
dtypes: float64(3), object(2)
memory usage: 624.3+ KB


In [344]:
# since bath and bhk are int values than the float so median is a good option

#since 0.874 is the corealation between bath and bhk(previously size) so can use bath to fill the missing bhk values.
df['bhk']=df['bhk'].fillna(df.groupby(df['bath'])['bhk'].transform('median'))

In [345]:

df['bhk'].isna().sum()

16

In [346]:
bhk_bath_null_rows=df[df['bhk'].isna() & df['bath'].isna()]
bhk_bath_null_rows

,location,total_sqft,bath,price,bhk
579,Sarjapur Road,1200 - 2400,NaN,34.185,NaN
1775,IVC Road,2000 - 5634,NaN,124.000,NaN
2264,Banashankari,2400,NaN,460.000,NaN
2809,Sarjapur Road,1200 - 2400,NaN,28.785,NaN
2862,Devanahalli,1500 - 2400,NaN,46.800,NaN
5333,Devanahalli,2100 - 5405,NaN,177.115,NaN
6423,Whitefield,2324,NaN,26.730,NaN
6636,Jigani,1500,NaN,25.490,NaN
6719,Hoskote,800 - 2660,NaN,28.545,NaN
7680,Kasavanhalli,5000,NaN,400.000,NaN


In [347]:
# 16 rows with bhk(size) as nan value also has nan bath value. so filling nan by universal median value  

df['bhk']=df['bhk'].fillna(df['bhk'].median())
df['bhk'].isna().sum()

0

In [348]:
# now using bhk values to fill 73 na values of bath. bath will be replace by the median of those columns which has same bhk values.

df['bath']=df['bath'].fillna(df.groupby('bhk')['bath'].transform('median'))
df['bath'].isna().sum()

0

In [349]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13319 entries, 0 to 13319
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   location    13319 non-null  object 
 1   total_sqft  13319 non-null  object 
 2   bath        13319 non-null  float64
 3   price       13319 non-null  float64
 4   bhk         13319 non-null  float64
dtypes: float64(3), object(2)
memory usage: 624.3+ KB


## further analysis


In [350]:
df[df['bhk']>20]

# These are basically outliers

,location,total_sqft,bath,price,bhk
1718,2Electronic City Phase II,8000,27.0,230.0,27.0
4684,Munnekollal,2400,40.0,660.0,43.0


In [351]:
df['total_sqft'].unique()

array(['1056', '2600', '1440', ..., '1133 - 1384', '774', '4689'],
      dtype=object)

In [352]:
def convertRange(x):
    temp = str(x).split("-")
    try:
        if len(temp) == 2:
            return (float(temp[0]) + float(temp[1])) / 2
        return float(x)
    except (ValueError,TypeError):
        return None

In [353]:
df['total_sqft']=df['total_sqft'].apply(convertRange)
df

,location,total_sqft,bath,price,bhk
0,Electronic City Phase II,1056.0,2.0,39.07,2.0
1,Chikka Tirupathi,2600.0,5.0,120.00,4.0
2,Uttarahalli,1440.0,2.0,62.00,3.0
3,Lingadheeranahalli,1521.0,3.0,95.00,3.0
4,Kothanur,1200.0,2.0,51.00,2.0
...,...,...,...,...,...
13315,Whitefield,3453.0,4.0,231.00,5.0
13316,Richards Town,3600.0,5.0,400.00,4.0
13317,Raja Rajeshwari Nagar,1141.0,2.0,60.00,2.0
13318,Padmanabhanagar,4689.0,4.0,488.00,4.0


In [354]:
# price per square feet and price is in lakhs

df['price_per_sqft']=(df['price']*100000)/df['total_sqft']
df

,location,total_sqft,bath,price,bhk,price_per_sqft
0,Electronic City Phase II,1056.0,2.0,39.07,2.0,3699.810606
1,Chikka Tirupathi,2600.0,5.0,120.00,4.0,4615.384615
2,Uttarahalli,1440.0,2.0,62.00,3.0,4305.555556
3,Lingadheeranahalli,1521.0,3.0,95.00,3.0,6245.890861
4,Kothanur,1200.0,2.0,51.00,2.0,4250.000000
...,...,...,...,...,...,...
13315,Whitefield,3453.0,4.0,231.00,5.0,6689.834926
13316,Richards Town,3600.0,5.0,400.00,4.0,11111.111111
13317,Raja Rajeshwari Nagar,1141.0,2.0,60.00,2.0,5258.545136
13318,Padmanabhanagar,4689.0,4.0,488.00,4.0,10407.336319


In [355]:
df.describe()

,total_sqft,bath,price,bhk,price_per_sqft
count,13273.000000,13319.000000,13319.000000,13319.000000,1.327300e+04
mean,1559.623652,2.695247,112.567621,2.803964,7.907692e+03
std,1238.451862,1.340006,148.977089,1.294261,1.064336e+05
min,1.000000,1.000000,8.000000,1.000000,2.678298e+02
25%,1100.000000,2.000000,50.000000,2.000000,4.266667e+03
50%,1276.000000,2.000000,72.000000,3.000000,5.434783e+03
75%,1680.000000,3.000000,120.000000,3.000000,7.311828e+03
max,52272.000000,40.000000,3600.000000,43.000000,1.200000e+07


In [356]:
df['location']=df['location'].apply(lambda x:x.strip())
location_count=df['location'].value_counts()
location_count

location
Whitefield            541
Sarjapur  Road        399
Electronic City       304
Kanakpura Road        273
Thanisandra           237
                     ... 
Hsr layout sector3      1
Thyagraj Nagar          1
Banagirinagar           1
Medaralli               1
Chikkajala              1
Name: count, Length: 1294, dtype: int64

In [357]:
location_count_less_10=location_count[location_count<=10]
location_count_less_10

location
Nagappa Reddy Layout     10
1st Block Koramangala    10
Naganathapura            10
Dodsworth Layout         10
Ganga Nagar              10
                         ..
Hsr layout sector3        1
Thyagraj Nagar            1
Banagirinagar             1
Medaralli                 1
Chikkajala                1
Name: count, Length: 1053, dtype: int64

In [358]:
df['location']=df['location'].apply(lambda x: 'other' if x in location_count_less_10 else x)
df['location'].value_counts()

location
other                        2885
Whitefield                    541
Sarjapur  Road                399
Electronic City               304
Kanakpura Road                273
                             ... 
Nehru Nagar                    11
Narayanapura                   11
Kodigehalli                    11
2nd Phase Judicial Layout      11
Marsur                         11
Name: count, Length: 242, dtype: int64

### Outlier detection and removal

In [359]:
df.describe()

,total_sqft,bath,price,bhk,price_per_sqft
count,13273.000000,13319.000000,13319.000000,13319.000000,1.327300e+04
mean,1559.623652,2.695247,112.567621,2.803964,7.907692e+03
std,1238.451862,1.340006,148.977089,1.294261,1.064336e+05
min,1.000000,1.000000,8.000000,1.000000,2.678298e+02
25%,1100.000000,2.000000,50.000000,2.000000,4.266667e+03
50%,1276.000000,2.000000,72.000000,3.000000,5.434783e+03
75%,1680.000000,3.000000,120.000000,3.000000,7.311828e+03
max,52272.000000,40.000000,3600.000000,43.000000,1.200000e+07


In [360]:
(df['total_sqft']/df['bhk']).describe()

count    13273.000000
mean       574.629098
std        387.586770
min          0.250000
25%        473.333333
50%        552.500000
75%        625.000000
max      26136.000000
dtype: float64

In [361]:
df=df[((df['total_sqft']/df['bhk'])>=300)]
df.describe()

,total_sqft,bath,price,bhk,price_per_sqft
count,12529.000000,12529.000000,12529.000000,12529.000000,12529.000000
mean,1594.564110,2.566366,111.384427,2.652087,6304.053504
std,1261.321632,1.080385,152.083230,0.976514,4162.395819
min,300.000000,1.000000,8.440000,1.000000,267.829813
25%,1116.000000,2.000000,49.000000,2.000000,4210.526316
50%,1300.000000,2.000000,70.000000,3.000000,5294.117647
75%,1700.000000,3.000000,115.000000,3.000000,6916.666667
max,52272.000000,16.000000,3600.000000,16.000000,176470.588235


In [362]:
df.shape

(12529, 6)

In [363]:
df['price_per_sqft'].describe()

count     12529.000000
mean       6304.053504
std        4162.395819
min         267.829813
25%        4210.526316
50%        5294.117647
75%        6916.666667
max      176470.588235
Name: price_per_sqft, dtype: float64

In [364]:
def remove_outliers_sqft(df):
    df_output=pd.DataFrame()

    for key,subdf in df.groupby('location'):
        m=np.mean(subdf['price_per_sqft'])
        sd= np.std(subdf['price_per_sqft'])

        gen_df=subdf[(subdf['price_per_sqft']> (m-sd)) & (subdf['price_per_sqft']< (m+sd))]
        df_output=pd.concat([df_output,gen_df],ignore_index=True)
    return df_output

df=remove_outliers_sqft(df)
df.describe()

,total_sqft,bath,price,bhk,price_per_sqft
count,10300.000000,10300.000000,10300.000000,10300.000000,10300.000000
mean,1508.431719,2.477864,91.286885,2.575340,5659.090455
std,880.736507,0.982809,86.346962,0.897642,2265.883017
min,300.000000,1.000000,10.000000,1.000000,1250.000000
25%,1110.000000,2.000000,49.000000,2.000000,4244.864208
50%,1286.000000,2.000000,67.000000,2.000000,5175.519668
75%,1650.000000,3.000000,100.000000,3.000000,6428.571429
max,30400.000000,16.000000,2200.000000,16.000000,24509.803922


In [365]:
def remove_outliers_bhk(df):
    exclude_indices=np.array([])
    for location,location_df in df.groupby('location'):
        bhk_stats={}
        for bhk,bhk_df in location_df.groupby('bhk'):
            bhk_stats[bhk]={
                'mean':np.mean(bhk_df.price_per_sqft),
                'sd':np.std(bhk_df.price_per_sqft),
                'count':bhk_df.shape[0]
            } 

        for bhk,bhk_df in location_df.groupby('bhk'):
            stats=bhk_stats.get(bhk-1)
            if stats and stats['count']>5:
                exclude_indices=np.append(exclude_indices,bhk_df[bhk_df.price_per_sqft<(stats['mean'])].index.values)

    return df.drop(exclude_indices,axis='index')

In [366]:
df=remove_outliers_bhk(df)

In [367]:
df.shape

(7360, 6)

In [368]:
df

,location,total_sqft,bath,price,bhk,price_per_sqft
0,1st Block Jayanagar,2850.0,4.0,428.0,4.0,15017.543860
1,1st Block Jayanagar,1630.0,3.0,194.0,3.0,11901.840491
2,1st Block Jayanagar,1875.0,2.0,235.0,3.0,12533.333333
3,1st Block Jayanagar,1200.0,2.0,130.0,3.0,10833.333333
4,1st Block Jayanagar,1235.0,2.0,148.0,2.0,11983.805668
...,...,...,...,...,...,...
10291,other,1200.0,2.0,70.0,2.0,5833.333333
10292,other,1800.0,1.0,200.0,1.0,11111.111111
10295,other,1353.0,2.0,110.0,2.0,8130.081301
10296,other,812.0,1.0,26.0,1.0,3201.970443


### Cleaned Data


In [369]:
# price per sqft was build only to detect outliers

df.drop(columns=['price_per_sqft'],inplace=True)
df

,location,total_sqft,bath,price,bhk
0,1st Block Jayanagar,2850.0,4.0,428.0,4.0
1,1st Block Jayanagar,1630.0,3.0,194.0,3.0
2,1st Block Jayanagar,1875.0,2.0,235.0,3.0
3,1st Block Jayanagar,1200.0,2.0,130.0,3.0
4,1st Block Jayanagar,1235.0,2.0,148.0,2.0
...,...,...,...,...,...
10291,other,1200.0,2.0,70.0,2.0
10292,other,1800.0,1.0,200.0,1.0
10295,other,1353.0,2.0,110.0,2.0
10296,other,812.0,1.0,26.0,1.0


In [370]:
df.to_csv("Cleaned_data.csv")

In [371]:
X=df.drop(columns=['price'])
X

,location,total_sqft,bath,bhk
0,1st Block Jayanagar,2850.0,4.0,4.0
1,1st Block Jayanagar,1630.0,3.0,3.0
2,1st Block Jayanagar,1875.0,2.0,3.0
3,1st Block Jayanagar,1200.0,2.0,3.0
4,1st Block Jayanagar,1235.0,2.0,2.0
...,...,...,...,...
10291,other,1200.0,2.0,2.0
10292,other,1800.0,1.0,1.0
10295,other,1353.0,2.0,2.0
10296,other,812.0,1.0,1.0


In [372]:
y=df['price']
y

0        428.0
1        194.0
2        235.0
3        130.0
4        148.0
         ...  
10291     70.0
10292    200.0
10295    110.0
10296     26.0
10299    400.0
Name: price, Length: 7360, dtype: float64

In [373]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,Lasso,Ridge
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

In [374]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=0)

In [375]:
print(X_train.shape)
print(X_test.shape)

(5888, 4)
(1472, 4)


In [376]:
X

,location,total_sqft,bath,bhk
0,1st Block Jayanagar,2850.0,4.0,4.0
1,1st Block Jayanagar,1630.0,3.0,3.0
2,1st Block Jayanagar,1875.0,2.0,3.0
3,1st Block Jayanagar,1200.0,2.0,3.0
4,1st Block Jayanagar,1235.0,2.0,2.0
...,...,...,...,...
10291,other,1200.0,2.0,2.0
10292,other,1800.0,1.0,1.0
10295,other,1353.0,2.0,2.0
10296,other,812.0,1.0,1.0


### Applying Linear Regression

In [377]:
preprocessor=ColumnTransformer(
    transformers=[
        ("cat",OneHotEncoder(handle_unknown="ignore"),['location']),
        ("num",StandardScaler(),['total_sqft','bath','bhk'])
    ],
    remainder="passthrough"
)

In [378]:
pipe=Pipeline([
    ("prep",preprocessor),
    ("lr",LinearRegression())
])

In [379]:
pipe.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('lr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['location','total_sqft','bath','bhk']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transfo

In [380]:
y_pred_lr=pipe.predict(X_test)
r2_score(y_test,y_pred_lr)

0.8543821748284752

### Lasso Regression

In [381]:
pipe=Pipeline([
    ("prep",preprocessor),
    ("lasso",Lasso())
])

In [382]:
pipe.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('lasso', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['location','total_sqft','bath','bhk']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the tran

In [383]:
y_pred_lasso=pipe.predict(X_test)
r2_score(y_test,y_pred_lasso)

0.7082441355759652

### Ridge Regression

In [384]:
pipe=Pipeline([
    ("prep",preprocessor),
    ("ridge",Ridge())
])

In [385]:
pipe.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('ridge', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['location','total_sqft','bath','bhk']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the tran

In [386]:
y_pred_ridge=pipe.predict(X_test)
r2_score(y_test,y_pred_ridge)

0.8524354434248399

In [387]:
import joblib

In [388]:
joblib.dump(pipe,'house_price_model.joblib')

['house_price_model.joblib']